In [1]:
# Run this cell first to install all required packages
import subprocess
import sys

packages = [
    'selenium',
    'webdriver-manager',
    'requests',
    'beautifulsoup4',
    'pandas',
    'numpy',
    'xgboost',
    'scikit-learn',
    'joblib',
    'fake-useragent',
    'openpyxl',
    'matplotlib',
    'tqdm'
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    print(f"✅ Installed: {package}")

print("\n✅ All packages installed successfully!")

✅ Installed: selenium
✅ Installed: webdriver-manager
✅ Installed: requests
✅ Installed: beautifulsoup4
✅ Installed: pandas
✅ Installed: numpy
✅ Installed: xgboost
✅ Installed: scikit-learn
✅ Installed: joblib
✅ Installed: fake-useragent
✅ Installed: openpyxl
✅ Installed: matplotlib
✅ Installed: tqdm

✅ All packages installed successfully!


In [2]:
import os
import re
import json
import time
import random
import logging
import warnings
import pickle
import tldextract
from bs4 import BeautifulSoup
from typing import Optional, Dict, List, Tuple
from urllib.parse import urlparse, urljoin
from datetime import datetime
from pathlib import Path

# Web scraping
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException

# For Chrome driver management
from webdriver_manager.chrome import ChromeDriverManager
import chromedriver_autoinstaller

# Data handling
import pandas as pd
import numpy as np

# ML imports
import joblib
import xgboost as xgb
from sklearn.preprocessing import StandardScaler

# Visualization (optional)
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Progress bar
from tqdm import tqdm

# Suppress warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful!")
print(f"📁 Current working directory: {os.getcwd()}")

✅ All imports successful!
📁 Current working directory: c:\Users\disha\OneDrive\Documents\company_website_predictor


In [8]:
# ==================== INPUT YOUR DATA HERE ====================

# 1️⃣ INPUT: Your Serper API Key (get from https://serper.dev/)
SERPER_API_KEY = "efa6cbe513613e548d0742a7e5cf6cab1ea82a01"  # <-- CHANGE THIS

# 2️⃣ INPUT: Your XGBoost Model Path
MODEL_PATH = "company_predictor_XGBmodel.joblib"  # <-- CHANGE THIS

# 3️⃣ INPUT: Load companies from CSV
csv_file = '500_companies.csv'  # <-- CHANGE THIS

# ==================== VERIFY MODEL ====================
print("🔍 Checking model file...")
if os.path.exists(MODEL_PATH):
    file_size = os.path.getsize(MODEL_PATH) / (1024 * 1024)
    print(f"✅ Model file found: {MODEL_PATH}")
    print(f"📦 File size: {file_size:.2f} MB")
    
    try:
        xgb_model = joblib.load(MODEL_PATH)
        print(f"✅ Model loaded successfully! Type: {type(xgb_model)}")
        
        if hasattr(xgb_model, 'n_features_in_'):
            print(f"📈 Expected features: {xgb_model.n_features_in_}")
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        raise
else:
    print(f"❌ Model file not found: {MODEL_PATH}")
    print("📂 Files in current directory:")
    for file in os.listdir('.'):
        if file.endswith(('.joblib', '.pkl', '.pickle')):
            print(f"  - {file} (model file found!)")
    raise FileNotFoundError(f"Model file {MODEL_PATH} not found")

# ==================== LOAD COMPANIES ====================
if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"\n📋 CSV Columns: {df.columns.tolist()}")
    print(f"📊 First 3 rows:")
    print(df.head(3))
    
    # IMPORTANT: Change 'company_name' to match your CSV column name
    # If your column is 'Company Name', use df['Company Name'].tolist()
    # If your column is 'company', use df['company'].tolist()
    # If your column is 'Name', use df['Name'].tolist()
    COMPANY_NAMES = df['Company Name'].tolist()  # <-- CHANGE COLUMN NAME
    
    print(f"\n✅ Loaded {len(COMPANY_NAMES)} companies from {csv_file}")
    print(f"📊 First 5 companies: {COMPANY_NAMES[:5]}")
else:
    print(f"❌ CSV file '{csv_file}' not found!")
    print("📂 CSV files in current directory:")
    csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    if csv_files:
        for f in csv_files:
            print(f"  - {f}")
    else:
        print("  No CSV files found!")
    
    # Fallback hardcoded list
    COMPANY_NAMES = [
        "Apple Inc.",
        "Microsoft Corporation", 
        "Google LLC",
        "Tesla Inc.",
        "Netflix Inc."
    ]
    print(f"\n⚠️  Using fallback list with {len(COMPANY_NAMES)} companies")

print("\n" + "="*50)
print("✅ All inputs ready!")
print(f"📊 Companies to process: {len(COMPANY_NAMES)}")
print(f"🔧 Model: {MODEL_PATH}")
print(f"📁 CSV file: {csv_file}")
print("="*50)
# ==================== ADD TO CELL 3 ====================
OUTPUT_DIR = "output_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
CHECKPOINT_INTERVAL = 50   # save checkpoint every N companies
LOG_FILE = os.path.join(OUTPUT_DIR, "pipeline.log")

🔍 Checking model file...
✅ Model file found: company_predictor_XGBmodel.joblib
📦 File size: 0.13 MB
✅ Model loaded successfully! Type: <class 'xgboost.sklearn.XGBClassifier'>
📈 Expected features: 4

📋 CSV Columns: ['Company Name']
📊 First 3 rows:
                Company Name
0  Saudi Arabian Oil Company
1                        KFC
2       A.P. Moller - Maersk

✅ Loaded 500 companies from 500_companies.csv
📊 First 5 companies: ['Saudi Arabian Oil Company', 'KFC', 'A.P. Moller - Maersk', 'NTT Corporation', 'Accor S.A.']

✅ All inputs ready!
📊 Companies to process: 500
🔧 Model: company_predictor_XGBmodel.joblib
📁 CSV file: 500_companies.csv


In [4]:
# ==================== CELL 4: LINKEDIN SCRAPER (IMPROVED) ====================
import time
import random
import re
import json
import tldextract
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

class LinkedInScraper:
    """Robust LinkedIn scraper with multiple extraction methods"""
    
    def __init__(self, headless: bool = True):
        self.headless = headless
        self.driver = None
        self.setup_logging()
        
    def setup_logging(self):
        logging.basicConfig(
            level=logging.WARNING,
            format='%(asctime)s - %(levelname)s - %(message)s'
        )
        self.logger = logging.getLogger(__name__)
    
    def create_driver(self) -> webdriver.Chrome:
        """Create Chrome driver with optimal anti-detection settings"""
        chromedriver_autoinstaller.install()
        
        chrome_options = Options()
        
        if self.headless:
            chrome_options.add_argument('--headless=new')
        
        # Anti-detection settings
        chrome_options.add_argument('--disable-blink-features=AutomationControlled')
        chrome_options.add_argument('--disable-dev-shm-usage')
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--disable-web-security')
        chrome_options.add_argument('--disable-gpu')
        chrome_options.add_argument('--disable-extensions')
        chrome_options.add_argument('--disable-popup-blocking')
        chrome_options.add_argument('--disable-notifications')
        chrome_options.add_argument('--disable-infobars')
        chrome_options.add_argument('--ignore-certificate-errors')
        chrome_options.add_argument('--window-size=1200,800')
        
        # Random user agent
        user_agents = [
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        ]
        chrome_options.add_argument(f'user-agent={random.choice(user_agents)}')
        
        chrome_options.add_experimental_option('excludeSwitches', ['enable-automation'])
        chrome_options.add_experimental_option('useAutomationExtension', False)
        chrome_options.add_experimental_option('prefs', {
            'profile.default_content_setting_values.notifications': 2,
            'credentials_enable_service': False,
            'profile.password_manager_enabled': False
        })
        
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
        
        driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
            'source': '''
                Object.defineProperty(navigator, 'webdriver', {
                    get: () => undefined
                });
                window.chrome = {
                    runtime: {}
                };
            '''
        })
        
        return driver
    
    def scrape_linkedin_profile(self, profile_url: str) -> Dict[str, Optional[str]]:
        """Main scraping method with multiple fallback strategies"""
        result = {
            'website': None,
            'company_name': None,
            'description': None,
            'industry': None,
            'employee_count': None,
            'headquarters': None
        }
        
        self.driver = self.create_driver()
        
        try:
            self.logger.info(f"Scraping LinkedIn profile: {profile_url}")
            
            self.driver.get(profile_url)
            time.sleep(random.uniform(3, 5))
            
            # Wait for page to load
            WebDriverWait(self.driver, 15).until(
                EC.presence_of_element_located((By.TAG_NAME, "body"))
            )
            
            # Scroll to load dynamic content
            self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            
            # Extract website using multiple methods
            website = self._extract_website_from_button()
            if website and website != 'static.licdn.com':
                result['website'] = website
                self.logger.info(f"Found website via button: {website}")
            
            if not result['website'] or result['website'] == 'static.licdn.com':
                website = self._extract_website_from_text()
                if website and website != 'static.licdn.com':
                    result['website'] = website
                    self.logger.info(f"Found website from text: {website}")
            
            if not result['website'] or result['website'] == 'static.licdn.com':
                website = self._extract_website_from_description()
                if website and website != 'static.licdn.com':
                    result['website'] = website
                    self.logger.info(f"Found website from description: {website}")
            
            if not result['website'] or result['website'] == 'static.licdn.com':
                website = self._extract_website_from_page_source()
                if website and website != 'static.licdn.com':
                    result['website'] = website
                    self.logger.info(f"Found website from page source: {website}")
            
            # Extract other info
            result['company_name'] = self._extract_company_name(profile_url)
            result['description'] = self._extract_description()
            result['industry'] = self._extract_industry()
            
            # Final check - if website is static.licdn.com, set to None
            if result['website'] == 'static.licdn.com':
                result['website'] = None
            
            return result
            
        except Exception as e:
            self.logger.error(f"Error scraping profile: {e}")
            return result
        finally:
            self.cleanup_driver()
    
    def _extract_website_from_button(self) -> Optional[str]:
        """Extract website from the website button/link"""
        try:
            # Look for the website button
            selectors = [
                '.org-page-details__website-link',
                '.link-without-visited-state',
                '.org-top-card__website-link',
                'a[data-anonymize="company-website"]',
                'a[href*="linkedin.com/redir/"]'
            ]
            
            for selector in selectors:
                try:
                    elements = self.driver.find_elements(By.CSS_SELECTOR, selector)
                    for element in elements:
                        href = element.get_attribute('href')
                        text = element.text.strip().lower()
                        
                        if href:
                            # Check if it's a website link
                            if 'website' in text or 'site' in text or 'www.' in href:
                                website = self._clean_website_url(href)
                                if website and website != 'static.licdn.com':
                                    return website
                except:
                    continue
            
            # Try finding by link text
            try:
                links = self.driver.find_elements(By.TAG_NAME, "a")
                for link in links:
                    text = link.text.strip().lower()
                    if 'website' in text or 'visit' in text and 'site' in text:
                        href = link.get_attribute('href')
                        if href:
                            website = self._clean_website_url(href)
                            if website and website != 'static.licdn.com':
                                return website
            except:
                pass
            
            return None
            
        except Exception as e:
            self.logger.error(f"Error extracting website from button: {e}")
            return None
    
    def _extract_website_from_text(self) -> Optional[str]:
        """Extract website from visible text on the page"""
        try:
            # Look for website in the company info section
            page_text = self.driver.find_element(By.TAG_NAME, "body").text
            
            # Look for website patterns
            patterns = [
                r'Website[:\s]*([a-zA-Z0-9-]+\.[a-zA-Z]{2,}(?:\.[a-zA-Z]{2,})?)',
                r'Visit our website[:\s]*([a-zA-Z0-9-]+\.[a-zA-Z]{2,}(?:\.[a-zA-Z]{2,})?)',
                r'Company Website[:\s]*([a-zA-Z0-9-]+\.[a-zA-Z]{2,}(?:\.[a-zA-Z]{2,})?)',
            ]
            
            for pattern in patterns:
                matches = re.findall(pattern, page_text, re.IGNORECASE)
                for match in matches:
                    website = self._clean_website_url(match)
                    if website and website != 'static.licdn.com':
                        return website
            
            return None
            
        except Exception as e:
            self.logger.error(f"Error extracting website from text: {e}")
            return None
    
    def _extract_website_from_description(self) -> Optional[str]:
        """Extract website from company description"""
        try:
            description = self._extract_description()
            if description:
                # Look for URL patterns in description
                url_pattern = r'(?:https?://)?(?:www\.)?([a-zA-Z0-9-]+\.[a-zA-Z]{2,}(?:\.[a-zA-Z]{2,})?)'
                matches = re.findall(url_pattern, description)
                for match in matches:
                    website = self._clean_website_url(match)
                    if website and website != 'static.licdn.com':
                        return website
            
            return None
            
        except Exception as e:
            self.logger.error(f"Error extracting website from description: {e}")
            return None
    
    def _extract_website_from_page_source(self) -> Optional[str]:
        """Extract website from page source"""
        try:
            page_source = self.driver.page_source
            soup = BeautifulSoup(page_source, 'html.parser')
            
            # Look for website in meta tags
            meta_tags = soup.find_all('meta')
            for meta in meta_tags:
                if meta.get('name') and 'website' in meta.get('name', '').lower():
                    content = meta.get('content', '')
                    if content:
                        website = self._clean_website_url(content)
                        if website and website != 'static.licdn.com':
                            return website
            
            # Look for JSON-LD
            script_tags = soup.find_all('script', {'type': 'application/ld+json'})
            for script in script_tags:
                try:
                    data = json.loads(script.string)
                    if isinstance(data, dict):
                        if 'url' in data:
                            website = self._clean_website_url(data['url'])
                            if website and website != 'static.licdn.com':
                                return website
                        if 'sameAs' in data:
                            for url in data['sameAs']:
                                website = self._clean_website_url(url)
                                if website and website != 'static.licdn.com':
                                    return website
                except:
                    continue
            
            return None
            
        except Exception as e:
            self.logger.error(f"Error extracting website from page source: {e}")
            return None
    
    def _extract_company_name(self, profile_url: str) -> Optional[str]:
        """Extract company name"""
        try:
            if 'linkedin.com/company/' in profile_url:
                name = profile_url.split('linkedin.com/company/')[-1].split('/')[0]
                name = name.replace('-', ' ').title()
                return name
            
            title = self.driver.title
            if ' | LinkedIn' in title:
                return title.replace(' | LinkedIn', '').strip()
            
            return None
        except:
            return None
    
    def _extract_description(self) -> Optional[str]:
        """Extract company description"""
        try:
            selectors = [
                '.org-page-details__description-text',
                '.break-words',
                '.org-top-card__description'
            ]
            
            for selector in selectors:
                try:
                    element = self.driver.find_element(By.CSS_SELECTOR, selector)
                    description = element.text.strip()
                    if description:
                        return description
                except:
                    continue
            
            return None
        except:
            return None
    
    def _extract_industry(self) -> Optional[str]:
        """Extract company industry"""
        try:
            selectors = ['.org-page-details__industry', '.org-top-card__industry']
            for selector in selectors:
                try:
                    element = self.driver.find_element(By.CSS_SELECTOR, selector)
                    return element.text.strip()
                except:
                    continue
            return None
        except:
            return None
    
    def _clean_website_url(self, url: str) -> Optional[str]:
        """Clean and validate website URL"""
        if not url:
            return None
        
        try:
            # Handle LinkedIn redirect URLs
            if 'linkedin.com/redir/' in url:
                redirect_match = re.search(r'url=([^&]+)', url)
                if redirect_match:
                    url = redirect_match.group(1)
            
            # Decode URL if needed
            if '%' in url:
                try:
                    url = requests.utils.unquote(url)
                except:
                    pass
            
            # Parse URL
            parsed = urlparse(url)
            
            # If no scheme, add http://
            if not parsed.scheme:
                url = f"http://{url}"
                parsed = urlparse(url)
            
            # Extract domain
            domain = parsed.netloc
            
            # Remove www prefix
            if domain.startswith('www.'):
                domain = domain[4:]
            
            # Remove trailing slashes
            domain = domain.rstrip('/')
            
            # Validate domain
            if self._is_valid_company_website(domain):
                return domain
            
            return None
            
        except Exception as e:
            self.logger.error(f"Error cleaning URL {url}: {e}")
            return None
    
    def _is_valid_company_website(self, domain: str) -> bool:
        """Validate if domain is likely a legitimate company website"""
        if not domain:
            return False
        
        domain_lower = domain.lower()
        
        # Reject common non-company domains
        blacklist = [
            'linkedin.com', 'facebook.com', 'twitter.com', 'instagram.com',
            'youtube.com', 'wikipedia.org', 'glassdoor.com', 'indeed.com',
            'crunchbase.com', 'bloomberg.com', 'reuters.com', 'businesswire.com',
            'prnewswire.com', 'marketwatch.com', 'forbes.com', 'cnbc.com',
            'static.licdn.com'  # <-- Add this to blacklist
        ]
        
        for blocked in blacklist:
            if blocked in domain_lower:
                return False
        
        # Must have at least one dot
        if '.' not in domain:
            return False
        
        # Not an IP address
        if re.match(r'^\d+\.\d+\.\d+\.\d+$', domain):
            return False
        
        # Must end with valid TLD
        valid_tlds = ['.com', '.org', '.net', '.io', '.co', '.uk', '.de', '.fr', '.cn', '.jp', '.in', '.ai']
        if not any(domain_lower.endswith(tld) for tld in valid_tlds):
            return False
        
        return True
    
    def cleanup_driver(self):
        """Clean up WebDriver"""
        if self.driver:
            try:
                self.driver.quit()
            except:
                pass
            self.driver = None
    
    def scrape_with_retry(self, profile_url: str, max_retries: int = 3) -> Dict[str, Optional[str]]:
        """Scrape with automatic retry"""
        for attempt in range(max_retries):
            try:
                result = self.scrape_linkedin_profile(profile_url)
                if result['website'] and result['website'] != 'static.licdn.com':
                    return result
                else:
                    print(f"   Attempt {attempt + 1} failed to find website")
                    if attempt < max_retries - 1:
                        wait_time = random.uniform(5, 10)
                        print(f"   Waiting {wait_time:.1f} seconds before retry...")
                        time.sleep(wait_time)
            except Exception as e:
                print(f"   Attempt {attempt + 1} error: {e}")
                if attempt < max_retries - 1:
                    time.sleep(random.uniform(5, 10))
        
        return {'website': None, 'company_name': None, 'description': None}

print("✅ LinkedIn Scraper loaded successfully!")

✅ LinkedIn Scraper loaded successfully!


In [5]:
# ==================== CELL 5: RESOLVER WITH RETRIES, CHECKPOINTS, LOGGING ====================
import time
import logging
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm import tqdm
import numpy as np
import pandas as pd
import re
from typing import Optional, Dict, List, Tuple
from urllib.parse import urlparse
import tldextract

class CompanyDomainResolver:
    def __init__(self, serper_api_key: str, xgboost_model):
        self.serper_api_key = serper_api_key
        self.xgboost_model = xgboost_model
        self.linkedin_scraper = LinkedInScraper(headless=True)
        # Setup requests session with retries
        self.session = requests.Session()
        retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
        self.session.mount('https://', HTTPAdapter(max_retries=retries))
        self.session.mount('http://', HTTPAdapter(max_retries=retries))
        self.setup_logging()

    def setup_logging(self):
        logging.basicConfig(level=logging.WARNING)
        self.logger = logging.getLogger(__name__)

    # ---------- Serper API with retries ----------
    def _serper_search(self, query: str, num: int = 10) -> dict:
        """Call Serper API with retries and exponential backoff."""
        params = {'api_key': self.serper_api_key, 'q': query, 'num': num}
        for attempt in range(5):
            try:
                resp = self.session.get('https://google.serper.dev/search', params=params, timeout=15)
                if resp.status_code == 429:
                    wait = (2 ** attempt) * 5
                    self.logger.warning(f"Rate limited. Waiting {wait}s")
                    time.sleep(wait)
                    continue
                resp.raise_for_status()
                return resp.json()
            except (requests.exceptions.RequestException, ValueError) as e:
                self.logger.warning(f"Serper attempt {attempt+1} failed: {e}")
                if attempt == 4:
                    raise
                time.sleep(2 ** attempt)
        return {}

    def get_linkedin_profile_url(self, company_name: str) -> Optional[str]:
        try:
            search_query = f"{company_name} LinkedIn"
            data = self._serper_search(search_query, num=5)
            for result in data.get('organic', []):
                link = result.get('link', '')
                if 'linkedin.com/company/' in link:
                    return link
            return None
        except Exception as e:
            self.logger.error(f"LinkedIn search error: {e}")
            return None

    def get_google_search_results(self, query: str, num_results: int = 10) -> List[Dict]:
        try:
            data = self._serper_search(query, num_results)
            return data.get('organic', [])
        except Exception as e:
            self.logger.error(f"Search error: {e}")
            return []

    # ---------- Helpers ----------
    def _extract_primary_domain(self, domain: str) -> Optional[str]:
        domain = domain.lower()
        prefixes = ['careers.', 'jobs.', 'hiring.', 'talent.', 'work.', 'recruiting.']
        for p in prefixes:
            if domain.startswith(p):
                domain = domain[len(p):]
                break
        if domain.endswith('careers.com'):
            base = domain.replace('careers.com', '')
            if base and len(base) >= 2:
                return base + '.com'
        return domain

    def _is_career_subdomain(self, domain: str) -> bool:
        prefixes = ['careers.', 'jobs.', 'hiring.', 'talent.', 'work.', 'recruiting.']
        for p in prefixes:
            if domain.startswith(p):
                return True
        if domain.endswith('careers.com'):
            return True
        return False

    def _domain_matches_company(self, domain: str, company_name: str) -> bool:
        if not domain or not company_name:
            return False
        clean = re.sub(r'\b(inc|llc|ltd|corp|corporation|plc|limited|group|holdings|the|co|company|international|global)\b', '', company_name, flags=re.I)
        clean = re.sub(r'[^a-zA-Z ]', '', clean).strip().lower()
        if not clean:
            clean = company_name.lower()
        extracted = tldextract.extract(domain)
        main = extracted.domain.lower()
        if not main:
            return False
        words = set(clean.split())
        if main in words:
            return True
        for w in words:
            if len(main) >= 3 and main in w:
                return True
            if len(w) >= 3 and w in main:
                return True
        acronym = ''.join([w[0] for w in clean.split() if w])
        if len(acronym) >= 2 and acronym == main:
            return True
        if len(main) <= 3 and clean.startswith(main):
            return True
        for w in words:
            if len(w) >= 3 and w in main:
                return True
        return False

    def _extract_domain(self, url: str) -> Optional[str]:
        if not url:
            return None
        parsed = urlparse(url)
        domain = parsed.netloc
        if domain.startswith('www.'):
            domain = domain[4:]
        return domain.rstrip('/') if domain else None

    def _is_valid_company_domain(self, domain: str) -> bool:
        if not domain:
            return False
        blacklist = ['linkedin.com', 'facebook.com', 'twitter.com', 'instagram.com',
                     'youtube.com', 'wikipedia.org', 'glassdoor.com', 'indeed.com',
                     'crunchbase.com', 'bloomberg.com', 'reuters.com', 'static.licdn.com']
        domain_lower = domain.lower()
        for b in blacklist:
            if b in domain_lower:
                return False
        if '.' not in domain:
            return False
        valid_tlds = ['.com', '.org', '.net', '.io', '.co', '.uk', '.de', '.fr', '.cn', '.jp', '.in', '.ai']
        return any(domain_lower.endswith(tld) for tld in valid_tlds)

    # ---------- ML resolve ----------
    def _ml_resolve(self, company_name: str) -> Tuple[Optional[str], str]:
        try:
            search_results = self.get_google_search_results(company_name)
            if not search_results:
                return None, 'ML_NoResults'
            for idx in range(min(3, len(search_results))):
                result = search_results[idx]
                features = self.prepare_ml_features_for_result(company_name, result, idx)
                if features is not None:
                    features_array = np.array(features).reshape(1, -1)
                    prediction = self.xgboost_model.predict(features_array)[0]
                    if prediction == 1:
                        link = result.get('link', '')
                        domain = self._extract_domain(link)
                        if domain and self._is_valid_company_domain(domain):
                            return (domain, 'ML_Fallback')
            # Fallback to first result
            link = search_results[0].get('link', '')
            domain = self._extract_domain(link)
            if domain:
                return (domain, 'ML_FirstResult')
            return None, 'ML_NoValidDomain'
        except Exception as e:
            self.logger.error(f"ML error: {e}")
            return None, 'ML_Error'

    def prepare_ml_features_for_result(self, company_name: str, result: Dict, position: int) -> Optional[List[float]]:
        try:
            link = result.get('link', '')
            domain = self._extract_domain(link)
            if not domain:
                return None
            pos = position + 1
            clean_name = re.sub(r'[^a-zA-Z0-9]', '', company_name.lower())
            clean_domain = re.sub(r'[^a-zA-Z0-9]', '', domain.lower())
            if clean_name and clean_domain:
                set1 = set(clean_name)
                set2 = set(clean_domain)
                similarity = len(set1 & set2) / len(set1 | set2) if (set1 | set2) else 0
            else:
                similarity = 0
            company_words = company_name.split()
            company_acronym = ''.join([w[0].lower() for w in company_words if w[0].isalpha()])
            domain_parts = domain.split('.')[0].split('-')
            domain_acronym = ''.join([p[0].lower() for p in domain_parts if p])
            acronym_match = 1 if company_acronym and domain_acronym and company_acronym == domain_acronym else 0
            social = ['linkedin', 'facebook', 'twitter', 'instagram', 'youtube', 'tiktok', 'snapchat']
            is_social = 1 if any(s in domain.lower() for s in social) else 0
            return [pos, similarity, acronym_match, is_social]
        except Exception as e:
            self.logger.error(f"Feature error: {e}")
            return None

    # ---------- Main resolve ----------
    def resolve_domain(self, company_name: str) -> Tuple[Optional[str], str, Dict]:
        # --- LinkedIn (always attempted) ---
        linkedin_domain = None
        linkedin_url = None
        try:
            linkedin_url = self.get_linkedin_profile_url(company_name)
            if linkedin_url:
                result = self.linkedin_scraper.scrape_with_retry(linkedin_url)
                raw = result.get('website')
                if raw and raw != 'static.licdn.com':
                    linkedin_domain = raw
        except Exception as e:
            self.logger.warning(f"LinkedIn failed: {e}")

        if linkedin_domain:
            if self._is_career_subdomain(linkedin_domain):
                primary = self._extract_primary_domain(linkedin_domain)
                if primary and self._domain_matches_company(primary, company_name):
                    return (primary, 'LinkedIn_Corrected', {'is_official': True, 'confidence': 85, 'reason': 'Career subdomain corrected'})
            else:
                if self._domain_matches_company(linkedin_domain, company_name):
                    return (linkedin_domain, 'LinkedIn_Validated', {'is_official': True, 'confidence': 85, 'reason': 'Domain matches company'})
                primary = self._extract_primary_domain(linkedin_domain)
                if primary and primary != linkedin_domain and self._domain_matches_company(primary, company_name):
                    return (primary, 'LinkedIn_Corrected', {'is_official': True, 'confidence': 75, 'reason': 'Extracted primary domain'})

        # --- ML fallback ---
        ml_domain, ml_source = self._ml_resolve(company_name)
        if ml_domain:
            return (ml_domain, ml_source, {'is_official': True, 'confidence': 70, 'reason': 'ML fallback'})

        return (None, 'Failed', {'is_official': False, 'confidence': 0, 'reason': 'No domain found'})

    # ---------- Batch processing with checkpoints and logging ----------
    def process_batch(self, company_names: List[str], checkpoint_interval: int = 50,
                      log_file: str = None) -> pd.DataFrame:
        rows = []
        total = len(company_names)

        # Setup file logging if provided
        if log_file:
            file_handler = logging.FileHandler(log_file, encoding='utf-8')
            file_handler.setLevel(logging.INFO)
            formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
            file_handler.setFormatter(formatter)
            self.logger.addHandler(file_handler)

        for idx, company in enumerate(tqdm(company_names, desc="Processing")):
            try:
                domain, source, validation = self.resolve_domain(company)
                self.logger.info(f"{company} -> {domain or 'None'} | Source: {source} | Conf: {validation.get('confidence',0):.1f}% | Official: {validation.get('is_official',False)}")
            except Exception as e:
                domain, source = None, 'Error'
                validation = {'is_official': False, 'confidence': 0, 'reason': str(e)}
                self.logger.error(f"{company} -> ERROR: {e}")

            rows.append({
                'Company': company,
                'Domain': domain,
                'Source': source,
                'IsOfficial': validation.get('is_official', False),
                'Confidence': validation.get('confidence', 0.0),
                'ValidationReason': validation.get('reason', ''),
                'PrimaryDomain': validation.get('primary_domain', '')
            })

            # Checkpoint
            if (idx + 1) % checkpoint_interval == 0:
                checkpoint_df = pd.DataFrame(rows)
                checkpoint_df.to_csv(f"checkpoint_{idx+1}.csv", index=False)
                self.logger.info(f"Checkpoint saved: {idx+1}/{total}")

            # Rate limit (2 seconds between companies)
            time.sleep(2)

        return pd.DataFrame(rows)

print("✅ Domain Resolver loaded successfully!")

✅ Domain Resolver loaded successfully!


In [9]:

# ==================== CELL 6: RUN PIPELINE ====================
import time
import logging

# Setup logging: errors to console, everything to file
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# File handler – detailed logs (everything)
file_handler = logging.FileHandler(LOG_FILE, encoding='utf-8')
file_handler.setLevel(logging.INFO)
file_formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler.setFormatter(file_formatter)
logger.addHandler(file_handler)

# Console handler – only warnings and errors
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.WARNING)
console_formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
console_handler.setFormatter(console_formatter)
logger.addHandler(console_handler)

print("🚀 Starting pipeline... (logs saved to", LOG_FILE, ")")
print("\n" + "="*60)
print("DOMAIN RESOLUTION PIPELINE")
print("="*60)

# Initialize resolver
resolver = CompanyDomainResolver(SERPER_API_KEY, xgb_model)

# Process companies
df_results = resolver.process_batch(
    COMPANY_NAMES,
    checkpoint_interval=5,
    log_file=LOG_FILE   # this will still be used to add the file handler in process_batch
)

# Display summary
print("\n" + "="*60)
print("📊 RESULTS SUMMARY")
print("="*60)
success_count = df_results['Domain'].notna().sum()
print(f"Total companies: {len(df_results)}")
print(f"Successfully resolved: {success_count}")
if 'IsOfficial' in df_results.columns:
    official = df_results['IsOfficial'].sum()
    print(f"Officially validated: {official}")

# Merge with original CSV if available
if 'df' in locals():
    try:
        # Try to find the correct column name
        possible_cols = ['company_name', 'Company', 'Name', 'Organization', 'Company Name']
        merge_col = None
        for col in possible_cols:
            if col in df.columns:
                merge_col = col
                break
        if merge_col:
            df_final = df.merge(df_results, left_on=merge_col, right_on='Company', how='left')
            print(f"✅ Merged with original CSV data using column: '{merge_col}'")
        else:
            print("⚠️ No matching column found in CSV. Using results only.")
            df_final = df_results
    except Exception as e:
        print(f"⚠️ Merge failed: {e}")
        df_final = df_results
else:
    df_final = df_results

print("\n✅ Pipeline finished. Results available in df_final")

🚀 Starting pipeline... (logs saved to output_results\pipeline.log )

DOMAIN RESOLUTION PIPELINE


Processing:   0%|          | 0/500 [00:00<?, ?it/s]

   Attempt 1 error: <urlopen error [Errno 11001] getaddrinfo failed>
   Attempt 2 error: <urlopen error [Errno 11001] getaddrinfo failed>
   Attempt 3 error: <urlopen error [Errno 11001] getaddrinfo failed>


2026-07-08 09:22:54,254 - INFO - Saudi Arabian Oil Company -> americas.aramco.com | Source: ML_Fallback | Conf: 70.0% | Official: True
Processing:   0%|          | 1/500 [00:56<7:50:05, 56.52s/it]

   Attempt 1 error: <urlopen error [Errno 11001] getaddrinfo failed>
   Attempt 2 error: <urlopen error [Errno 11001] getaddrinfo failed>
   Attempt 3 error: <urlopen error [Errno 11001] getaddrinfo failed>


...

   Attempt 2 failed to find website
   Waiting 9.1 seconds before retry...


2026-07-08 12:53:00,931 - INFO - Chromedriver is already installed.
2026-07-08 12:53:00,932 - INFO - ====== WebDriver manager ======
2026-07-08 12:53:02,001 - INFO - Driver [C:\Users\disha\.wdm\drivers\chromedriver\win64\149.0.7827.155\chromedriver-win64/chromedriver.exe] found in cache by browser version
2026-07-08 12:53:03,257 - INFO - Scraping LinkedIn profile: https://au.linkedin.com/company/commonwealthbank


   Attempt 3 failed to find website


2026-07-08 12:53:25,718 - INFO - Commonwealth Bank of Australia -> commbank.com.au | Source: ML_FirstResult | Conf: 70.0% | Official: True
Processing:  96%|█████████▌| 479/500 [3:31:27<14:36, 41.74s/it]2026-07-08 12:53:29,209 - INFO - Chromedriver is already installed.
2026-07-08 12:53:29,212 - INFO - ====== WebDriver manager ======
2026-07-08 12:53:30,307 - INFO - Driver [C:\Users\disha\.wdm\drivers\chromedriver\win64\149.0.7827.155\chromedriver-win64/chromedriver.exe] found in cache by browser version
2026-07-08 12:53:31,546 - INFO - Scraping LinkedIn profile: https://www.linkedin.com/company/icbc_2
2026-07-08 12:53:50,750 - INFO - Found website via button: icbc.com.cn
2026-07-08 12:53:54,222 - INFO - Industrial and Commercial Bank of China -> icbc.com.cn | Source: ML_Fallback | Conf: 70.0% | Official: True
2026-07-08 12:53:54,249 - INFO - Checkpoint saved: 480/500
Processing:  96%|█████████▌| 480/500 [3:31:56<12:35, 37.78s/it]2026-07-08 12:53:58,883 - INFO - Chromedriver is already 


📊 RESULTS SUMMARY
Total companies: 500
Successfully resolved: 500
Officially validated: 500
✅ Merged with original CSV data using column: 'Company Name'

✅ Pipeline finished. Results available in df_final


In [ ]:
# ==================== CELL 7: SAVE RESULTS ====================
import os
from datetime import datetime

def save_results(df, prefix="company_domains", output_dir="."):
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{prefix}_{timestamp}.csv"
    filepath = os.path.join(output_dir, filename)
    df.to_csv(filepath, index=False)
    print(f"✅ Saved to: {filepath}")
    return filepath

# Save full and simple versions
save_results(df_final, prefix="company_domains_full", output_dir=OUTPUT_DIR)
save_results(df_final[['Company', 'Domain', 'Source', 'Confidence']], prefix="company_domains_simple", output_dir=OUTPUT_DIR)

✅ Saved to: .\company_domains_20260707_105417.csv
   Total companies: 20
   Successfully resolved: 19
   Officially validated: 19
✅ Saved to: .\company_domains_simple_20260707_105417.csv
   Total companies: 20
   Successfully resolved: 19

✅ All CSV files saved successfully!
